# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [3]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.ensemble import AdaBoostClassifier

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
df=spaceship
df.shape

(8693, 14)

In [5]:
df.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [6]:
df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [7]:
df.dropna(inplace=True)

In [8]:
df['Cabin'] = df['Cabin'].str[0]
df['Cabin'].value_counts()

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

In [9]:
#eliminar columnas PassengerId, Name
df.drop(columns=['PassengerId','Name'],inplace=True)

In [10]:
df['Destination']=df['Destination'].str.strip()
df=pd.get_dummies(df,columns=['Destination'],drop_first=True)

In [ ]:
df=pd.get_dummies(df,columns=['CryoSleep'],drop_first=True)

In [11]:
df=pd.get_dummies(df,columns=['VIP'],drop_first=True)

In [12]:
#hacer dummies de la columna HomePlanet
df=pd.get_dummies(df,columns=['HomePlanet'],drop_first=True)

In [13]:
df=pd.get_dummies(df,columns=['Cabin'],drop_first=True)

In [16]:
df['CryoSleep_True']=df['CryoSleep'].astype('int')
df['VIP_True']=df['VIP_True'].astype('int')
df['HomePlanet_Europa']=df['HomePlanet_Europa'].astype('int')
df['HomePlanet_Mars']=df['HomePlanet_Mars'].astype('int')
df['Destination_PSO J318.5-22']=df['Destination_PSO J318.5-22'].astype('int')
df['Destination_TRAPPIST-1e']=df['Destination_TRAPPIST-1e'].astype('int')
df['Transported']=df['Transported'].astype('int')

In [17]:
cabin_columns = ['Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T']
df[cabin_columns] = df[cabin_columns].astype(int)


In [18]:
X = df.drop(columns=['Transported'])
y = df['Transported']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [19]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb.score(X_test, y_test)

0.8071104387291982

In [20]:
 #precision
y_pred_gb_= gb.predict(X_test)
precision_score(y_test, y_pred_gb_, average=None)

array([0.83728814, 0.78278689])

In [21]:
#recall
recall_score(y_test, y_pred_gb_, average=None)

array([0.75650842, 0.85650224])

In [22]:
#f1
f1_score(y_test, y_pred_gb_, average=None)

array([0.79485117, 0.81798715])

- Evaluate your model

In [24]:
#clasiification report
print(classification_report(y_test, y_pred_gb_))

              precision    recall  f1-score   support

           0       0.84      0.76      0.79       653
           1       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [26]:
# uso de gridsearch
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7]
}







- Run Grid Search

In [27]:
gb = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gb, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
grid_search.best_params_

Fitting 3 folds for each of 27 candidates, totalling 81 fits


{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

In [28]:
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb.score(X_test, y_test)


0.8086232980332829

- Evaluate your model

In [29]:
#precision
y_pred_gb = gb.predict(X_test)
precision_score(y_test, y_pred_gb, average=None)


array([0.830033  , 0.79050279])

In [30]:
#recall
recall_score(y_test, y_pred_gb, average=None)


array([0.77029096, 0.84603886])

In [31]:
#f1
f1_score(y_test, y_pred_gb, average=None)

array([0.79904686, 0.81732852])

In [32]:
#clasiification report
print(classification_report(y_test, y_pred_gb))


              precision    recall  f1-score   support

           0       0.83      0.77      0.80       653
           1       0.79      0.85      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



In [33]:
# El modelo no mejora con el uso de gridsearch